Nama : Rangga Saputra

NIM : 250401020034

Kelas : IF405

---

# Praktikum Pertemuan 3 Data Science

In [3]:
# ========================================
# PIPELINE DATA CLEANING - HOUSING DATASET
# Created by Rangga Saputra (250401020034)
# ========================================

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

Penjelasan:
- pandas : mengolah data
- numpy : operasi numerik
- winsorize : sebenarnya untuk outlier, walau nanti tugas memakai IQR Fence

In [4]:
# Load Dataset & Eksplorasi Awal
df = pd.read_csv('housing_dirty.csv')
print("Shape awal:", df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

Shape awal: (130, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.450000e+02    2.000000   1991.25

Tujuannya:
- melihat tipe data
- melihat jumlah missing value
- memahami kondisi dataset sebelum dibersihkan

In [5]:
# Hapus Duplikat
df.drop_duplicates(inplace=True)
print("Shape setelah hapus duplikat:", df.shape)

Shape setelah hapus duplikat: (130, 7)


Penjelasan:
- drop_duplicates() : menghapus baris yang sama persis
- inplace=True : langsung mengubah dataframe asli

In [6]:
# Normalisasi String
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# Imputasi Missing Value
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

- Karena ada data seperti: "jakarta, Jakarta, JAKARTA", maka diseragamkan menjadi "Jakarta".
- "Baik" menjadi "baik".
- Untuk numerik menggunakan median, agar lebih aman terhadap outlier dibanding rata-rata.

In [7]:
# Tangani Outlier dengan IQR
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

Penjelasan:
- Q1 : kuartil bawah
- Q3 : kuartil atas
- IQR : jarak antar kuartil
- clip() : membatasi nilai agar tidak melewati batas bawah dan atas

In [8]:
# Validasi
assert df.isnull().sum().sum() == 0, "Masih ada missing value!"
assert df.duplicated().sum() == 0, "Masih ada data duplikat!"
print("Dataset sudah bersih!")

# Export CSV
df.to_csv('housing_clean.csv', index=False)
print("housing_clean.csv berhasil disimpan!")

Dataset sudah bersih!
housing_clean.csv berhasil disimpan!


Validasi Dataset
- Pastikan dataset sudah bersih.
- Cek duplikat data.
- Simpan Dataset Bersih "housing_clean.csv"

In [9]:
# Akses API JSONPlaceholder
url = 'https://jsonplaceholder.typicode.com/posts'
api_df = pd.read_json(url)
print(api_df.head())

   userId  id                                              title  \
0       1   1  sunt aut facere repellat provident occaecati e...   
1       1   2                                       qui est esse   
2       1   3  ea molestias quasi exercitationem repellat qui...   
3       1   4                               eum et est occaecati   
4       1   5                                 nesciunt quas odio   

                                                body  
0  quia et suscipit\nsuscipit recusandae consequu...  
1  est rerum tempore vitae\nsequi sint nihil repr...  
2  et iusto sed quo iure\nvoluptatem occaecati om...  
3  ullam et saepe reiciendis voluptatem adipisci\...  
4  repudiandae veniam quaerat sunt sed\nalias aut...  


Penjelasan:
- mengambil data dari API publik
- mengubah JSON menjadi DataFrame